In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
nltk.download('stopwords')
import re
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
nltk.download('omw-1.4')
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity



[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [9]:
documents = [
    "Machine learning is a branch of artificial intelligence that enables computers to learn from data.",

    "Supervised learning trains a model using labeled training data.",

    "Unsupervised learning discovers patterns and structures in unlabeled data.",

    "Classification is a machine learning task that assigns data to predefined categories.",

    "Regression predicts continuous numerical values from input features.",

    "Natural language processing enables computers to process and understand human language.",

    "Text preprocessing prepares raw text for machine learning by cleaning and transforming it.",

    "Tokenization divides text into smaller units such as words or tokens.",

    "Stop word removal eliminates common words that may provide little useful information.",

    "Stemming reduces words to their basic stems by removing prefixes or suffixes.",

    "Lemmatization converts words into their meaningful dictionary base forms.",

    "TF-IDF measures how important a word is to a document within a collection of documents.",

    "Cosine similarity measures the similarity between two vectors based on the angle between them.",

    "Semantic search attempts to find information based on meaning rather than only exact keywords.",

    "Word embeddings represent words as numerical vectors that capture relationships between their meanings.",

    "Neural networks are machine learning models inspired by the structure of biological neural networks.",

    "Deep learning uses neural networks with multiple layers to learn complex patterns from large datasets.",

    "Overfitting occurs when a model learns the training data too closely and performs poorly on unseen data.",

    "Training data is used to teach a machine learning model how to make predictions or decisions.",

    "Model evaluation uses metrics and test data to measure how well a machine learning model performs."
]



In [22]:
class PreprocessingModule:
    def __init__(self):
        self.stop_words = set(stopwords.words("english"))
        self.lemmatizer = WordNetLemmatizer()



    def transform(self, text):
         tokens = word_tokenize(text.lower())
         tokens = [self.lemmatizer.lemmatize(token) for token in tokens
                if re.match(r'^[a-zA-Z-]+$', token) and token not in self.stop_words]
         return tokens


pre = PreprocessingModule()
print(pre.transform("How does TF-IDF work?"))


['tf-idf', 'work']


In [23]:
class VectorizerModule:
    def __init__(self):
        self.vectorizer = TfidfVectorizer(stop_words='english')
        self.corpus_vectors = None  # will be set after fit()

    def fit(self, corpus):
        """
        Learns vocabulary from the corpus and stores TF-IDF vectors.
        Parameters: corpus (list of str) - the documents to index
        Returns: None
        """
        self.corpus_vectors = self.vectorizer.fit_transform(corpus)

    def transform(self, query):
        """
        Converts a new query into a TF-IDF vector using the already-learned vocabulary.
        Parameters: query (str) - a new piece of text to vectorize
        Returns: sparse vector representation of the query
        """
        query_vector = self.vectorizer.transform([query])
        return query_vector

vec = VectorizerModule()
vec.fit(["I love dogs", "I love cats", "The weather is sunny"])
result = vec.transform("dogs are great")
print(result.shape)


(1, 5)


In [24]:
class Pipeline:
    def __init__(self):
        self.preprocessor = PreprocessingModule()
        self.vectorizer = VectorizerModule()

    def run(self, query, corpus, top_k=3):
        """..."""
        cleaned_corpus = [" ".join(self.preprocessor.transform(doc)) for doc in corpus]
        self.vectorizer.fit(cleaned_corpus)

        cleaned_query = " ".join(self.preprocessor.transform(query))   # ← THIS must come first

        if len(cleaned_query.strip()) == 0:                             # ← THEN this check
               raise ValueError("Query has no valid words after cleaning. Please try a different query.")

        query_vector = self.vectorizer.transform(cleaned_query)
        scores = cosine_similarity(query_vector, self.vectorizer.corpus_vectors)
        top_indices = np.argsort(scores[0])[::-1][:top_k]
        results = [(scores[0][i], corpus[i]) for i in top_indices]
        return results

pipeline = Pipeline()
results = pipeline.run("How does TF-IDF work?", documents, top_k=3)
print(results)

for score, sentence in results:
    print(f"{score:.4f} - {sentence}")

[(np.float64(0.47084907815651583), 'TF-IDF measures how important a word is to a document within a collection of documents.'), (np.float64(0.0), 'Model evaluation uses metrics and test data to measure how well a machine learning model performs.'), (np.float64(0.0), 'Overfitting occurs when a model learns the training data too closely and performs poorly on unseen data.')]
0.4708 - TF-IDF measures how important a word is to a document within a collection of documents.
0.0000 - Model evaluation uses metrics and test data to measure how well a machine learning model performs.
0.0000 - Overfitting occurs when a model learns the training data too closely and performs poorly on unseen data.


In [25]:
def retrieve(query, pipeline, corpus, top_k=3, threshold=0.1):
    """
    Retrieve the most relevant documents for a query.

    Parameters:
        query: User's search query
        pipeline: Day 10 Pipeline instance
        corpus: Knowledge base
        top_k: Number of results to return
        threshold: Minimum similarity score

    Returns:
        Relevant results or a message when no relevant document exists.
    """

    results = pipeline.run(query, corpus, top_k=top_k)

    if results[0][0] < threshold:
        return "No relevant document found."

    return results

In [26]:
pipeline = Pipeline()

# Should trigger threshold (out-of-domain)
print(retrieve("What is the best pizza topping?", pipeline, documents))

print()

# Should return real results (in-domain)
print(retrieve("How does TF-IDF work?", pipeline, documents))


No relevant document found.

[(np.float64(0.47084907815651583), 'TF-IDF measures how important a word is to a document within a collection of documents.'), (np.float64(0.0), 'Model evaluation uses metrics and test data to measure how well a machine learning model performs.'), (np.float64(0.0), 'Overfitting occurs when a model learns the training data too closely and performs poorly on unseen data.')]


In [27]:
queries = [
    # 1–5: Clear in-domain queries
    "What is tokenization?",
    "How does cosine similarity work?",
    "What is supervised learning?",
    "What is TF-IDF?",
    "What is lemmatization?",

    # 6–7: Ambiguous queries
    "What is a model?",
    "How does learning work?",

    # 8–9: Completely out-of-domain queries
    "What's the weather today?",
    "What are the best places to visit in Italy?",

    # 10: Additional in-domain query
    "What is overfitting?"
]

for q in queries:
    print(f"\n=== Query: {q} ===")

    result = retrieve(
        q,
        pipeline,
        documents,
        top_k=3
    )

    if isinstance(result, str):
        print(result)
    else:
        for score, sentence in result:
            print(f"{score:.4f} - {sentence}")


=== Query: What is tokenization? ===
0.4028 - Tokenization divides text into smaller units such as words or tokens.
0.0000 - Model evaluation uses metrics and test data to measure how well a machine learning model performs.
0.0000 - Overfitting occurs when a model learns the training data too closely and performs poorly on unseen data.

=== Query: How does cosine similarity work? ===
0.7420 - Cosine similarity measures the similarity between two vectors based on the angle between them.
0.0000 - Model evaluation uses metrics and test data to measure how well a machine learning model performs.
0.0000 - Overfitting occurs when a model learns the training data too closely and performs poorly on unseen data.

=== Query: What is supervised learning? ===
0.4723 - Supervised learning trains a model using labeled training data.
0.1055 - Unsupervised learning discovers patterns and structures in unlabeled data.
0.0978 - Classification is a machine learning task that assigns data to predefined c